In [4]:
import time
import subprocess
import torch
from torch.utils.data import DataLoader
from transformers import GPT2Tokenizer
from datasets import load_dataset
from models.gpt2 import build_gpt2
from models.loss import gpt2_loss


def prepare_dataset(tokenizer, split="train", seq_len=256):
    ds = load_dataset("wikitext", "wikitext-2-raw-v1")[split]

    def encode(ex):
        tok = tokenizer(
            ex["text"],
            truncation=True,
            padding="max_length",
            max_length=seq_len,
        )
        return {
            "input_ids": tok["input_ids"],
            "attention_mask": tok["attention_mask"],
        }

    ds = ds.map(
        encode,
        batched=True,
        remove_columns=["text"],
        load_from_cache_file=True,
    )
    ds.set_format(type="torch", columns=["input_ids", "attention_mask"])
    return ds


def gpu_stats():
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=utilization.gpu,temperature.gpu,power.draw", "--format=csv,noheader"],
        capture_output=True, text=True,
    ).stdout.strip()
    return out


def train_single_gpu(num_epochs=1, bs=16, lr=2e-5):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"device: {device}")

    tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
    tokenizer.pad_token = tokenizer.eos_token

    model = build_gpt2(n_positions=256).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, eps=1e-5)
    scaler = torch.cuda.amp.GradScaler()

    loader = DataLoader(
        prepare_dataset(tokenizer, "train"),
        batch_size=bs,
        shuffle=True,
        num_workers=0,
        pin_memory=False,
        prefetch_factor=None,
    )

    print(f"model memory: {torch.cuda.memory_allocated()/1024**3:.2f}GB")
    print(f"steps per epoch: {len(loader)}")

    torch.cuda.reset_peak_memory_stats()

    total_samples = 0
    peak_mem_gb = 0.0

    for epoch in range(num_epochs):
        model.train()
        epoch_start = time.perf_counter()
        window_start = time.perf_counter()
        window_samples = 0

        for i, batch in enumerate(loader):
            input_ids = batch["input_ids"].to(device)
            attn = batch["attention_mask"].to(device)

            opt.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda", dtype=torch.float16):
                outputs = model(input_ids, attention_mask=attn, labels=input_ids)
                loss = gpt2_loss(outputs)

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()

            window_samples += bs
            total_samples += bs

            if i % 20 == 0 and i > 0:
                torch.cuda.synchronize()
                elapsed = time.perf_counter() - window_start
                throughput = window_samples / elapsed

                mem_gb = torch.cuda.max_memory_allocated() / 1024**3
                peak_mem_gb = max(peak_mem_gb, mem_gb)

                print(
                    f"epoch={epoch} step={i:>5d} loss={loss.item():.4f} "
                    f"throughput={throughput:.1f} samples/s mem={mem_gb:.2f}GB | "
                    f"gpu: {gpu_stats()}"
                )

                window_start = time.perf_counter()
                window_samples = 0

        torch.cuda.synchronize()
        epoch_time = time.perf_counter() - epoch_start
        avg_throughput = total_samples / epoch_time

        print(
            f"\n=== epoch {epoch} done | "
            f"time={epoch_time:.1f}s | "
            f"avg_throughput={avg_throughput:.1f} samples/s | "
            f"peak_mem={peak_mem_gb:.2f}GB ===\n"
        )


if __name__ == "__main__":
    train_single_gpu(num_epochs=1, bs=16, lr=2e-5)

device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Map:   0%|          | 0/36718 [00:00<?, ? examples/s]

model memory: 0.48GB
steps per epoch: 2295


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


epoch=0 step=   20 loss=2.4989 throughput=36.0 samples/s mem=7.56GB | gpu: 100 %, 47, 59.74 W
epoch=0 step=   40 loss=1.3994 throughput=39.2 samples/s mem=7.56GB | gpu: 99 %, 51, 41.72 W
epoch=0 step=   60 loss=1.6343 throughput=39.2 samples/s mem=7.56GB | gpu: 100 %, 54, 59.45 W
epoch=0 step=   80 loss=2.5759 throughput=38.9 samples/s mem=7.56GB | gpu: 100 %, 57, 69.25 W
epoch=0 step=  100 loss=2.4317 throughput=38.8 samples/s mem=7.56GB | gpu: 100 %, 59, 61.66 W
epoch=0 step=  120 loss=2.3528 throughput=38.6 samples/s mem=7.56GB | gpu: 100 %, 61, 59.09 W
epoch=0 step=  140 loss=1.7189 throughput=38.4 samples/s mem=7.56GB | gpu: 100 %, 64, 42.12 W
epoch=0 step=  160 loss=2.3894 throughput=38.1 samples/s mem=7.56GB | gpu: 100 %, 66, 47.73 W
epoch=0 step=  180 loss=2.2315 throughput=37.9 samples/s mem=7.56GB | gpu: 100 %, 68, 59.45 W
epoch=0 step=  200 loss=1.3952 throughput=37.6 samples/s mem=7.56GB | gpu: 100 %, 70, 68.75 W
epoch=0 step=  220 loss=2.7048 throughput=37.4 samples/s mem=

In [2]:
!git -C /content/training pull 2>/dev/null || git clone https://github.com/krishnajha23/training.git /content/training
%cd /content/training
!ls models/

Cloning into '/content/training'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 44 (delta 8), reused 40 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (44/44), 45.50 KiB | 5.69 MiB/s, done.
Resolving deltas: 100% (8/8), done.
/content/training
gpt2_jax.py  gpt2.py  loss.py  two_tower.py


In [3]:
import torch
import gc

# kill everything holding GPU memory
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

print(torch.__version__)
print(torch.cuda.get_device_name(0))
print(f"free: {torch.cuda.mem_get_info()[0]/1024**3:.2f}GB")
print(f"total: {torch.cuda.mem_get_info()[1]/1024**3:.2f}GB")
x = torch.randn(4, 256, 768).cuda()
with torch.amp.autocast("cuda", dtype=torch.float16):
    y = torch.nn.Linear(768, 768).cuda()(x)
    print(f"autocast output dtype: {y.dtype}")  # should be torch.float16
print(f"GPU utilization check:")
print(f"  allocated: {torch.cuda.memory_allocated()/1024**3:.2f}GB")
print(f"  reserved:  {torch.cuda.memory_reserved()/1024**3:.2f}GB")

2.10.0+cu128
Tesla T4
free: 14.46GB
total: 14.56GB
autocast output dtype: torch.float16
GPU utilization check:
  allocated: 0.02GB
  reserved:  0.02GB


In [ ]:
import time
import subprocess
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from models.two_tower import TwoTowerModel
from models.loss import in_batch_contrastive_loss


class SyntheticTwoTowerDataset(Dataset):
    """
    Synthetic dataset for two-tower benchmarking.
    Produces random user and item feature vectors of the correct shape.
    Real Amazon Reviews would have the same dims — swap this out if needed.

    user_feature_dim = 32 (embedding) + 2 (avg_rating, review_count) = 34
    item_feature_dim = 32 (embedding) + 2 (avg_rating, review_count) = 34
    """
    def __init__(self, num_samples=50000, user_feature_dim=34, item_feature_dim=34):
        self.num_samples = num_samples
        self.user_features = torch.randn(num_samples, user_feature_dim)
        self.item_features = torch.randn(num_samples, item_feature_dim)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        return {
            "user_features": self.user_features[idx],
            "item_features": self.item_features[idx],
        }


def gpu_stats():
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=utilization.gpu,temperature.gpu,power.draw",
         "--format=csv,noheader"],
        capture_output=True, text=True,
    ).stdout.strip()
    return out


def train_two_tower_gpu(
    num_epochs=1,
    bs=256,
    lr=1e-3,
    num_samples=50000,
    user_feature_dim=34,
    item_feature_dim=34,
    hidden_dims=(256, 128),
    embed_dim=64,
    temperature=0.07,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"device: {device}")

    model = TwoTowerModel(
        user_feature_dim=user_feature_dim,
        item_feature_dim=item_feature_dim,
        hidden_dims=list(hidden_dims),
        embed_dim=embed_dim,
    ).to(device)

    total_params = sum(p.numel() for p in model.parameters())
    print(f"two-tower parameters: {total_params/1e6:.2f}M")

    opt = torch.optim.AdamW(model.parameters(), lr=lr, eps=1e-5)
    scaler = torch.amp.GradScaler("cuda")

    dataset = SyntheticTwoTowerDataset(num_samples, user_feature_dim, item_feature_dim)
    loader = DataLoader(
        dataset,
        batch_size=bs,
        shuffle=True,
        num_workers=0,
        pin_memory=False,
        prefetch_factor=None,
    )

    torch.cuda.reset_peak_memory_stats()
    print(f"model memory: {torch.cuda.memory_allocated()/1024**3:.2f}GB")
    print(f"steps per epoch: {len(loader)}")

    total_samples = 0
    peak_mem_gb = 0.0

    for epoch in range(num_epochs):
        model.train()
        epoch_start = time.perf_counter()
        window_start = time.perf_counter()
        window_samples = 0

        for i, batch in enumerate(loader):
            user_features = batch["user_features"].to(device)
            item_features = batch["item_features"].to(device)

            opt.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda", dtype=torch.float16):
                user_emb, item_emb = model(user_features, item_features)
                loss = in_batch_contrastive_loss(user_emb, item_emb, temperature)

            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()

            window_samples += bs
            total_samples += bs

            if i % 20 == 0 and i > 0:
                torch.cuda.synchronize()
                elapsed = time.perf_counter() - window_start
                throughput = window_samples / elapsed

                mem_gb = torch.cuda.max_memory_allocated() / 1024**3
                peak_mem_gb = max(peak_mem_gb, mem_gb)

                print(
                    f"epoch={epoch} step={i:>5d} loss={loss.item():.4f} "
                    f"throughput={throughput:.1f} samples/s mem={mem_gb:.2f}GB | "
                    f"gpu: {gpu_stats()}"
                )

                window_start = time.perf_counter()
                window_samples = 0

        torch.cuda.synchronize()
        epoch_time = time.perf_counter() - epoch_start
        avg_throughput = total_samples / epoch_time

        print(
            f"\n=== epoch {epoch} done | "
            f"time={epoch_time:.1f}s | "
            f"avg_throughput={avg_throughput:.1f} samples/s | "
            f"peak_mem={peak_mem_gb:.2f}GB ===\n"
        )


if __name__ == "__main__":
    train_two_tower_gpu(num_epochs=1, bs=256, lr=1e-3)